# 🧥 MeshVTON — 3D Inference / Virtual Try-On

Eğitilmiş **ControlNet3D** ağırlıklarını kullanarak gerçek 3D-aware try-on yapar:

**kişi görseli + 3D kıyafet mesh'i (.obj)** → SMPL-X beden → drape → render (RGB+normal+depth) → ControlNet3D → sonuç

> ⚠️ Senin novel katkın ControlNet3D'dir ve **sadece 3D yol** (`.obj` mesh) bunu kullanır.
> 2D yol (düz kıyafet fotoğrafı) ControlNet3D'yi atlar — bu notebook'ta sadece karşılaştırma için var.

Gereksinim: GPU runtime + pytorch3d.

## 1️⃣ GPU Kontrol

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU yok! Runtime → Change runtime type → GPU')

## 2️⃣ Repo'yu klonla

In [ ]:
import os
REPO_URL = 'https://github.com/SerhanTelatar/MeshVTON.git'
PROJECT_DIR = '/content/MeshVTON'
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 3️⃣ Kütüphaneler (eğitimle AYNI sürümler) + 3D bağımlılıkları

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# IDM-VTON ile uyumlu SABİT sürümler
!pip install diffusers==0.25.0 transformers==4.36.2 accelerate==0.25.0 huggingface_hub==0.20.3 peft==0.7.1 -q
!pip install -q omegaconf opencv-python-headless pillow scipy einops timm controlnet_aux

# 3D pipeline — pytorch3d ŞART (mesh render için)
from google.colab import drive
drive.mount('/content/drive')
!pip install -q fvcore iopath smplx trimesh
!pip install -q /content/drive/MyDrive/wheels/pytorch3d*.whl
try:
    import pytorch3d
    print(f'✅ pytorch3d {pytorch3d.__version__}')
except Exception as e:
    print(f'❌ pytorch3d yüklenemedi ({e}) — 3D yol çalışmaz. Wheel Drive/MyDrive/wheels altında mı?')

## 4️⃣ SMPL-X modelini hazırla

3D yol, kişi görselinden beden çıkarmak için SMPL-X model dosyalarına ihtiyaç duyar.
Bunlar Drive'daki `pretrained.zip` içinde — çıkarıp `checkpoints/pretrained/smplx/` altına yerleştiriyoruz.

In [ ]:
import zipfile, shutil, subprocess
from pathlib import Path

P = Path('/content/MeshVTON')
DRIVE_DATA = '/content/drive/MyDrive/MeshVTON'

ckpt_dir = P / 'checkpoints'
ckpt_dir.mkdir(parents=True, exist_ok=True)

pre_zip = Path(DRIVE_DATA) / 'pretrained.zip'
smplx_target = P / 'checkpoints/pretrained/smplx'
smplx_target.mkdir(parents=True, exist_ok=True)

if (smplx_target / 'SMPLX_NEUTRAL.npz').exists():
    print('⏩ SMPL-X zaten hazır')
elif pre_zip.exists():
    print('📦 pretrained.zip açılıyor...')
    with zipfile.ZipFile(pre_zip) as z:
        z.extractall(ckpt_dir)
    # .npz dosyalarını bul ve doğru isimle yerleştir
    res = subprocess.run(['find', str(ckpt_dir), '-name', '*.npz'], capture_output=True, text=True)
    for line in res.stdout.strip().split('\n'):
        if not line.strip():
            continue
        src = Path(line)
        dest = smplx_target / ('SMPLX_NEUTRAL_2020.npz' if 'NEUTRAL_2020' in src.name else 'SMPLX_NEUTRAL.npz')
        if src.resolve() != dest.resolve():
            shutil.copy2(src, dest)
    print('✅ SMPL-X hazır:', (smplx_target / 'SMPLX_NEUTRAL.npz').exists())
else:
    print(f'❌ {pre_zip} bulunamadı — Drive yolunu kontrol et')

## 5️⃣ Eğitilmiş checkpoint'i bul

In [ ]:
from pathlib import Path

search_dirs = [
    Path('/content/MeshVTON/checkpoints/runs'),
    Path(DRIVE_DATA) / 'checkpoints',
    Path(DRIVE_DATA) / 'checkpoints/runs',
]
candidates = []
for d in search_dirs:
    if d.exists():
        candidates += sorted(d.glob('*.pt'), key=lambda p: p.stat().st_mtime)

if candidates:
    CKPT_PATH = str(candidates[-1])
    for c in candidates:
        print(f'  {c}  ({c.stat().st_size/1e9:.2f} GB)')
    print(f'\n✅ Kullanılacak: {CKPT_PATH}')
else:
    CKPT_PATH = None
    print('❌ Checkpoint yok. Eğitimdeki "Save to Drive" hücresini çalıştırdın mı?')

## 6️⃣ Pipeline + ControlNet3D ağırlıkları

In [ ]:
import sys, torch
sys.path.insert(0, '/content/MeshVTON')
from src.models.tryon_pipeline import TryOnPipeline

pipeline = TryOnPipeline.from_pretrained(
    pretrained_model_id='yisol/IDM-VTON',
    controlnet_3d_channels=9,
)

if CKPT_PATH:
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
    state = ckpt.get('model', ckpt)
    prefix = 'controlnet_3d.'
    cn_state = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
    if cn_state:
        missing, unexpected = pipeline.controlnet_3d.load_state_dict(cn_state, strict=False)
        print(f'✅ ControlNet3D yüklendi: {len(cn_state)} tensör (missing={len(missing)}, unexpected={len(unexpected)})')
    else:
        print('⚠️ controlnet_3d.* anahtarı yok — eğitilmemiş ağırlık.')
else:
    print('⚠️ Checkpoint yok — eğitilmemiş ControlNet3D.')

pipeline = pipeline.to('cuda').eval()

# pytorch3d rasterizer fp16 DESTEKLEMEZ → 3D render float32 ister.
# Büyük GPU'da en güvenlisi tüm modeli fp32 yapmak (autocast'e gerek kalmaz).
# (Küçük VRAM'de bu satırı kaldırıp fp16 + autocast kullan.)
pipeline = pipeline.float()
print('✅ Pipeline hazır (fp32)')

## 7️⃣ ImageTryOn oluştur

In [ ]:
from src.inference.image_tryon import ImageTryOn

tryon = ImageTryOn(
    pipeline=pipeline,
    config={
        'device': 'cuda',
        'image': {'resolution': 512},
        'sampling': {'num_inference_steps': 30, 'guidance_scale': 2.0, 'seed': 42},
        'postprocess': {'face_restore': True, 'edge_smooth': True, 'color_correction': True},
    },
)
print('✅ ImageTryOn hazır')

## 8️⃣ Girdileri yükle: kişi görseli + 3D kıyafet mesh'i

Önce **kişi görseli** (.jpg/.png), sonra **3D kıyafet mesh'i** (.obj) yükle.

`.obj` mesh'in yoksa: eğitim verisindeki `garments_3d.zip` içinden bir tane alabilirsin
(örn. `upper_body/*.obj`).

In [ ]:
from google.colab import files

print('👤 KİŞİ görselini yükle:')
person_up = files.upload()
PERSON_PATH = list(person_up.keys())[0]

print('\n🧵 3D KIYAFET mesh (.obj) yükle:')
mesh_up = files.upload()
GARMENT_MESH = list(mesh_up.keys())[0]

print(f'\nPerson: {PERSON_PATH}\nMesh:   {GARMENT_MESH}')

## 9️⃣ 3D Try-On çalıştır (ControlNet3D — senin katkın)

SMPL-X → drape → render → ControlNet3D → diffusion. `view_angle`: 0=ön, 90=yan, 180=arka.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Not: pipeline fp32 olduğu için autocast YOK — render (pytorch3d) fp32 çalışır.
result = tryon.run_with_3d_garment(
    PERSON_PATH, GARMENT_MESH,
    output_path='result_3d.png',
    view_angle=0.0,
)

fig, ax = plt.subplots(1, 2, figsize=(11, 6))
ax[0].imshow(Image.open(PERSON_PATH)); ax[0].set_title('Kişi'); ax[0].axis('off')
ax[1].imshow(result); ax[1].set_title('3D Try-On (ControlNet3D)'); ax[1].axis('off')
plt.tight_layout(); plt.savefig('result_3d_compare.png', dpi=120, bbox_inches='tight'); plt.show()
print('✅ result_3d.png kaydedildi')

## 🔟 (Opsiyonel) 2D karşılaştırma — ControlNet3D OLMADAN

Aynı kişiye düz bir kıyafet **fotoğrafı** (.jpg/.png) ile saf IDM-VTON backbone'u uygular
(3D koşullandırma yok). 3D sonuçla farkı görmek için.

In [ ]:
# from google.colab import files
# import torch
# from PIL import Image
# print('👕 Düz kıyafet FOTOĞRAFI (.jpg/.png) yükle:')
# g = files.upload(); GARMENT_IMG = list(g.keys())[0]
#
# with torch.autocast('cuda', dtype=torch.float16):
#     result2d = tryon.run(PERSON_PATH, GARMENT_IMG, output_path='result_2d.png')
# display(result2d)